# Customize Tuner for Neural Network Optimization

In this notebook, we explore how to customize hyperparameter optimization for a model:
- Use different hyperparameter selections or settings for the standard models
- Optimize only part of a pipeline (e.g. tune only the classifier while keeping the embedder fixed) using `PipelineWithHyperparameterRooting` and a custom `AbstractMotherTuner`
- Define a custom model and use it in hyperparameter optimization

## Key concepts from `mother.optimization.core`

### `AbstractMotherTuner`
An abstract base class for building custom tuners. Subclasses must implement two methods:
- **`objective(trial, context)`** — defines a single Optuna trial: samples hyperparameters, trains and evaluates the model, and returns a scalar score.
- **`call_optimize(context)`** — controls the overall optimization loop, i.e. how `study.optimize()` is called with `self.objective`.

The `optimize()` method (inherited from `AbstractMotherTuner`) orchestrates the full workflow: it creates the Optuna study, assembles the `ObjectiveContext`, enqueues any default parameters, and finally retrains the best model on the full dataset.

### `ObjectiveContext`
A dataclass that bundles all arguments from `optimize()` into a single object passed to `objective()` and `call_optimize()`. Its key fields are:

| Field | Description |
|---|---|
| `get_hyper_space` | Callable that, given a trial, returns a hyperparameter dict |
| `estimator` | The unfitted pipeline to be tuned |
| `X` / `y` | Training data and targets |
| `cross_validation` | Cross-validator (e.g. `KFold`) |
| `groups` | Optional group labels for grouped CV |
| `fit_kwargs` | Extra keyword arguments forwarded to `estimator.fit()` |
| `extras` | Arbitrary additional data your custom tuner may need |

In [4]:
%load_ext autoreload
%autoreload 2
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from torch import nn
from six import iteritems

from optuna import Trial
import sklearn.base as skl_base
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score


import mother.optimization as opt

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2)
print(
    f"Train / Validation samples = {X_train.shape[0]} / {X_valid.shape[0]} with {X_train.shape[1]} features and {y_train.nunique()} labels."
)

/workspaces/MotherML/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train / Validation samples = 455 / 114 with 30 features and 2 labels.


## 1. Optimize Part of a Pipeline

With a custom optimizer, you can optimize only a part of your pipeline.

Here, we optimize the classification model in a pipeline consisting of `TabPFNEmbeddingTransformer` (to generate model-based embeddings from the input features) and `CatboostClassifier`.

### 1.1 Create a Pipeline

In [5]:
from mother.ml.models.m_tabpfn import TabPFNEmbeddingTransformer
from mother.ml.models.m_catboost import CatboostClassifierMother
from mother.ml import PipelineWithHyperparameterRooting

model = PipelineWithHyperparameterRooting(
    [
        (
            "embedder",
            TabPFNEmbeddingTransformer(
                model_type="classification",
                use_kfold=False,  # these parameters will not change after optimization
            ),
        ),
        ("classifier", CatboostClassifierMother(target_type="single_target", logging_level="Silent")),
    ]
)

model.steps

[('embedder', TabPFNEmbeddingTransformer(use_kfold=False)),
 ('classifier',
  <mother.ml.models.m_catboost.CatboostClassifierMother at 0x716bad735940>)]

In an sklearn pipeline, parameter names indicate which step they belong to using the convention `{step_name}__{parameter_name}`.

In [6]:
model.get_params()

{'memory': None,
 'steps': [('embedder', TabPFNEmbeddingTransformer(use_kfold=False)),
  ('classifier',
   <mother.ml.models.m_catboost.CatboostClassifierMother at 0x716bad735940>)],
 'verbose': False,
 'embedder': TabPFNEmbeddingTransformer(use_kfold=False),
 'classifier': <mother.ml.models.m_catboost.CatboostClassifierMother at 0x716bad735940>,
 'embedder__device': 'cpu',
 'embedder__embedding_column_name': 'tabpfnembedding',
 'embedder__ignore_pretraining_limits': True,
 'embedder__model': None,
 'embedder__model_type': 'classification',
 'embedder__n_folds': 5,
 'embedder__random_state': None,
 'embedder__return_separate_columns': True,
 'embedder__use_kfold': False,
 'classifier__learning_rate': 0.03,
 'classifier__loss_function': 'Logloss',
 'classifier__logging_level': 'Silent',
 'classifier__auto_class_weights': 'Balanced',
 'classifier__random_strength': 1,
 'classifier__boosting_type': 'Plain',
 'classifier__bootstrap_type': 'Bayesian',
 'classifier__max_depth': 6,
 'classifi

### 1.2 Custom Optimizer

In the custom `objective` function, we select only parameters whose names start with `classifier`, so that only the classifier is optimized while the embedder remains fixed.

In [7]:
class CustomMotherTuner(opt.AbstractMotherTuner):
    def __init__(self, **kwargs):
        # create a customised scorer
        super().__init__(**kwargs)

    def objective(self, trial: Trial, context: opt.ObjectiveContext) -> float:
        cv_score = 0.0

        suggested_params_to_train: dict = context.get_hyper_space(trial=trial, X=context.X, y=context.y)

        # select params only for "classifier"
        suggested_params_to_train = {k: v for k, v in suggested_params_to_train.items() if k.startswith("classifier")}

        print(f"Trial {trial.number} parameters to tune : {suggested_params_to_train}")

        for train_idx, test_idx in context.cross_validation.split(context.X, context.y):
            # Train
            pipeline = skl_base.clone(context.estimator)

            # fit
            pipeline.set_params(**suggested_params_to_train)
            pipeline.fit(context.X.iloc[train_idx, :], context.y.iloc[train_idx])

            # Test score
            y_pred_test = pipeline.predict(X=context.X.iloc[test_idx, :])
            cv_score += accuracy_score(context.y.iloc[test_idx], y_pred_test)

        return cv_score / (context.cross_validation.get_n_splits())  # mean acc

    def call_optimize(self, context: opt.ObjectiveContext) -> None:
        self.study.optimize(
            lambda trial: self.objective(trial, context=context),
            n_trials=self.n_trials_optuna,
            gc_after_trial=True,
            callbacks=self.get_callbacks(),
        )


tuner = CustomMotherTuner(
    n_trials_optuna=3,  # number of trials for hyperparameter optimization
    n_threads_optuna=10,  # parallel threads for cross-validation evaluation
    n_startup_trials=1,  # number of random trials before using optuna
    tuning_direction="maximize",  # Maximize the accuracy
)

model_tuned = tuner.optimize(
    model,
    X_train.iloc[:100, :],
    y_train.iloc[:100],
    cross_validation=KFold(n_splits=2),
    hyperparameter_space_function=model.get_hyperparameter_space,
)

/workspaces/MotherML/src/mother/optimization/core.py:135: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:135: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:135: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(


Trial 0 parameters to tune : {'classifier__bootstrap_type': 'Bayesian', 'classifier__learning_rate': 0.03, 'classifier__random_strength': 1, 'classifier__grow_policy': 'SymmetricTree', 'classifier__max_depth': 6, 'classifier__loss_function': 'Logloss'}
Trial 1 parameters to tune : {'classifier__bootstrap_type': 'Bayesian', 'classifier__learning_rate': 0.02208007163250455, 'classifier__random_strength': 1.1961172071448225, 'classifier__grow_policy': 'SymmetricTree', 'classifier__max_depth': 5, 'classifier__loss_function': 'Logloss'}
Trial 2 parameters to tune : {'classifier__bootstrap_type': 'Bayesian', 'classifier__learning_rate': 0.18132275158254602, 'classifier__random_strength': 0.8829987555087553, 'classifier__grow_policy': 'SymmetricTree', 'classifier__max_depth': 6, 'classifier__auto_class_weights': 'None', 'classifier__loss_function': 'Focal:focal_alpha=0.3924531071968482;focal_gamma=6.2963722761684995'}
{'classifier__bootstrap_type': 'Bayesian', 'classifier__learning_rate': 0.0

After optimization, we can verify that the embedder parameters remain unchanged.

In [8]:
model_tuned.get_params()["embedder__use_kfold"]

False

## 2. Mother Optimization for a PyTorch Model

In this example, we create a custom `torch` model wrapper using `AbstractMotherPipeline` and a custom optimizer for the model.

### 2.1 Torch Wrapper for Mother
In this example, we only tune the learning rate, but additional parameters can be added for optimization.

In [ ]:
from collections import OrderedDict
from mother.ml import AbstractMotherPipeline
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
import torch
from mother.ml.models.utils import add_prefix_to_dict_keys
from optuna.trial import Trial
import pandas as pd
from sklearn.metrics import accuracy_score


class TorchNNMother(AbstractMotherPipeline):
    def __init__(self, lr: float = 1e-3, **kwargs) -> None:

        self.network = nn.Sequential(
            OrderedDict(
                [
                    ("linear1", nn.Linear(in_features=30, out_features=10)),
                    ("relu", nn.ReLU()),
                    ("linear2", nn.Linear(in_features=10, out_features=1)),
                    ("sigmoid", nn.Sigmoid()),
                ]
            )
        )

        self.optimizer = Adam(self.network.parameters(), lr=lr)
        self.lr = lr
        self.loss = nn.BCELoss()
        self._init_params: dict = {"lr": lr}

        non_optimised_params: list[str] = ["_init_params"]
        for k, v in kwargs.items():
            if k not in non_optimised_params:
                self._init_params[k] = v

    def default_parameters(self, prefix: str = "") -> dict:
        return add_prefix_to_dict_keys({"lr": 1e-3}, prefix=prefix)

    def get_hyperparameter_space(self, X, y, trial: Trial, prefix: str = "") -> dict:
        # hyper parameter search for learning rate
        suggested_params: dict = {"lr": trial.suggest_float(prefix + "lr", 1e-5, 1e-3, log=True)}
        suggested_params = add_prefix_to_dict_keys(suggested_params, prefix=prefix)

        return suggested_params

    def get_params(self, deep=True) -> dict:
        return self._init_params

    def set_params(self, **params):
        for key, value in iteritems(params):
            if key in self._init_params.keys():
                self._init_params[key] = value

        # Keep runtime attributes and optimizer in sync with tuned params.
        self.lr = float(self._init_params["lr"])

        return self.__init__(**self._init_params)

    def _get_dataloader(self, X, y, batch_size: int = 128, shuffle: bool = False) -> DataLoader:
        # Crate a dataloader for torch training
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        return DataLoader(
            TensorDataset(
                torch.tensor(X).to(torch.float32),
                torch.tensor(y).reshape(-1, 1).to(torch.float32),
            ),
            batch_size=batch_size,
            shuffle=shuffle,
        )

    def validation(self, X_valid, y_valid):
        valid_data_loader = self._get_dataloader(X_valid, y_valid)

        self.network.eval()
        valid_loss = 0.0
        valid_acc = 0.0

        with torch.no_grad():
            for X, y in valid_data_loader:
                y_hat = self.network(X)
                valid_loss += self.loss(y_hat, y).item() / len(valid_data_loader)
                y_hat_pred = (y_hat > 0.5).float()
                valid_acc += accuracy_score(
                    y.detach().numpy().ravel(),
                    y_hat_pred.detach().numpy().ravel(),
                ) / len(valid_data_loader)

        return valid_loss, valid_acc

    def fit(self, X_train, y_train, n_epochs: int = 100):
        train_data_loader = self._get_dataloader(X_train, y_train, shuffle=True)

        # Recreate optimizer so each fit starts from current tuned lr and clean state.
        self.optimizer = Adam(self.network.parameters(), lr=float(self._init_params["lr"]))

        for epoch in range(n_epochs):
            self.network.train()
            train_loss = 0.0
            for X, y in train_data_loader:
                self.optimizer.zero_grad()
                y_hat = self.network(X)
                batch_train_loss = self.loss(y_hat, y)

                if not torch.isfinite(batch_train_loss):
                    raise ValueError("Training diverged (non-finite loss). Try a lower learning rate.")

                batch_train_loss.backward()
                self.optimizer.step()
                train_loss += batch_train_loss.item() / len(train_data_loader)
        return self


model = TorchNNMother(lr=1e-3)
model.fit(X_train, y_train)
print(model.validation(X_valid, y_valid))

(0.2540920674800873, 0.9035087719298246)


### 2.2 Custom Mother Tuner for Torch

Neural network training often requires splitting data into three sets (train, validation, and test). This does not align with the default `MotherTuner` design, which does not create a validation set for monitoring training epochs. To address this, we add a train-validation split inside the custom objective function.

In [ ]:
from optuna import Trial
import sklearn.base as skl_base
import mother.optimization as opt


class TorchMotherTuner(opt.AbstractMotherTuner):
    def __init__(self, **kwargs):
        # create a customised scorer
        super().__init__(**kwargs)

    def objective(self, trial: Trial, context: opt.ObjectiveContext) -> float:
        X_train, X_valid, y_train, y_valid = train_test_split(context.X, context.y, test_size=0.2)
        print(
            f"Train / Validation samples = {X_train.shape[0]} / {X_valid.shape[0]} with {X_train.shape[1]} features and {y_train.nunique()} labels."
        )

        # fit
        estimator = skl_base.clone(context.estimator)
        suggested_params_to_train: dict = context.get_hyper_space(trial=trial, X=context.X, y=context.y)
        estimator.set_params(**suggested_params_to_train)
        estimator.fit(X_train, y_train)

        # calculate valid loss
        valid_loss, valid_acc = context.estimator.validation(X_valid, y_valid)

        return valid_loss

    def call_optimize(self, context: opt.ObjectiveContext) -> None:
        self.study.optimize(
            lambda trial: self.objective(trial, context=context),
            n_trials=self.n_trials_optuna,
            gc_after_trial=True,
            callbacks=self.get_callbacks(),
        )


tuner = TorchMotherTuner(
    n_trials_optuna=3,  # number of trials for hyperparameter optimization
    n_threads_optuna=10,  # parallel threads for cross-validation evaluation
    n_startup_trials=1,  # number of random trials before using optuna
    tuning_direction="minimize",  # Need to minimize the loss!
)

model_tuned = tuner.optimize(
    model,
    X,
    y,
    cross_validation=None,
    hyperparameter_space_function=model.get_hyperparameter_space,
)

/workspaces/MotherML/src/mother/optimization/core.py:131: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:131: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:131: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(


Train / Validation samples = 455 / 114 with 30 features and 2 labels.
{'lr': 0.001}
Train / Validation samples = 455 / 114 with 30 features and 2 labels.
{'lr': 0.0009361658778024464}
Train / Validation samples = 455 / 114 with 30 features and 2 labels.
{'lr': 0.0008048266581936503}
FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.22355324029922485], datetime_start=datetime.datetime(2026, 7, 17, 14, 11, 33, 456914), datetime_complete=datetime.datetime(2026, 7, 17, 14, 11, 34, 358672), params={'lr': 0.001}, user_attrs={}, system_attrs={'fixed_params': {'lr': 0.001}}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.001, log=True, low=1e-05, step=None)}, trial_id=0, value=None)
{'lr': 0.001}
{'lr': 0.001}
